# 3D coculture image segmentation

Updated the microsam pipeline to segment 3D image stack.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import os
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
root_logger = logging.getLogger()
for handler in root_logger.handlers[:]:
    root_logger.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from image_processing_tools.image_class.micro_sam_pipeline import load_config
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET145/01",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET145/02",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET145/03",
    )

search_path = (
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/03",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/04",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/05",
)

config_path = Path("./micro_sam_config.json")

file_pattern = ("CET*DAPI*.tif",
                "CET*Cy5*.tif",
                "*DIC*.tif",
                "MAX_CET145*")

config = load_config(config_path)
config['base_input_dir'] = Path("~/A8/Data_Roan/").expanduser()
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
config['model_type'] = 'vit_l_lm'
config['segmentation_mode'] = 'automatic'

config['preprocessing']['resize_image'] = False
config['preprocessing']['max_dim'] = 1024
config['preprocessing']['quantization'] = "8bit"
config['preprocessing']['correct_DIC_shift'] = [0,0]
config['ndim'] = 3
# config['tiling']['tile_shape'] = (512,512)
# config['tiling']['halo'] = (64,64)
config['segmentation_3d']['seed_slice'] = 49
config['segmentation_3d']['projection'] = "mask"

In [ ]:
dic_files = []
for p in search_path:
    dic_files.extend(find_files_by_pattern(p, file_pattern[2], verbose=True))
    
dic_imgs = [ImageContainer(img,config) for img in dic_files]

In [ ]:
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline
pipeline = MicroSAMPipeline(dic_imgs,config=config)
pipeline.run()

In [ ]:
pipeline.save_results()

## Two channels

In [ ]:
# Find the files. The 'files' variable will hold the list of Path objects.
dapi_files = []
cy5_files = []
for p in search_path: 
    dapi_files.append(find_files_by_pattern(p, file_pattern[0], verbose=True))
    cy5_files.append(find_files_by_pattern(p, file_pattern[1], verbose=True))

# dapi_imgs = [ImageContainer(img,config) for img in dapi_files]
# dic_imgs = [ImageContainer(img,config) for img in dic_files]

# merged_imgs = [dapi+dic for dapi,dic in zip(dapi_imgs, dic_imgs)]

In [ ]:
dapi_container = ImageContainer(dapi_files[0],config=config)
cy5_container = ImageContainer(cy5_files[0],config=config)
combined = cy5_container + dapi_container
combined.dapi_channel_index = 1
combined.dapi_slice = 44

In [ ]:
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline

p = MicroSAMPipeline(combined,config=config)
p.run()

In [ ]:
p.save_results()